# Tuen Mun hourly AQHI cleaning and quality checks
Read the 15 unmodified EPD English monthly CSVs in `data/raw`. Run all cells from the repository root with a Python 3 kernel and pandas installed. Output files are written under `data/processed`.

The timestamp is the Hong Kong hour endpoint (UTC+08:00): source hour 01 is 01:00 on the source date and hour 24 is midnight the following day. Daily Max rows are excluded.

In [1]:
from pathlib import Path
from datetime import date, datetime, timedelta
import csv
import re

import pandas as pd

raw_dir = Path("data/raw")
output_dir = Path("data/processed")
months = pd.period_range("2025-04", "2026-06", freq="M")
rows, date_hour_issues, invalid_aqhi = [], [], []

for month in months:
    path = raw_dir / f"{month.strftime('%Y%m')}_Eng.csv"
    current_date = None
    previous_hour = None
    with path.open(encoding="utf-8-sig", newline="") as source:
        records = csv.reader(source, strict=True)
        header = next(row for row in records if row[:2] == ["Date", "Hour"])
        station_column = header.index("Tuen Mun")
        for row in records:
            if not row or all(not cell.strip() for cell in row):
                continue
            location = f"{path.name}:{records.line_num}"
            if len(row) != len(header):
                date_hour_issues.append((location, "Unexpected column count", row))
                continue
            date_text, hour_text = (cell.strip() for cell in row[:2])
            if hour_text == "Daily Max":
                continue

            if date_text:
                last_date = current_date
                current_date = None
                try:
                    if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", date_text):
                        raise ValueError
                    current_date = date.fromisoformat(date_text)
                    if (current_date.year, current_date.month) != (month.year, month.month):
                        raise ValueError
                except ValueError:
                    current_date = None
                    date_hour_issues.append((location, "Invalid date or wrong file month", date_text))
                    continue
                if last_date is not None and current_date != last_date + timedelta(days=1):
                    date_hour_issues.append((location, "Unexpected date-block order", date_text))

            if current_date is None or not re.fullmatch(r"0[1-9]|[1-9]|1[0-9]|2[0-4]", hour_text):
                date_hour_issues.append((location, "Missing date context or invalid hour", hour_text))
                continue
            hour = int(hour_text)
            if date_text and hour != 1 or not date_text and (
                previous_hour is None or previous_hour == 24 or hour != previous_hour + 1
            ):
                date_hour_issues.append((location, "Unexpected hour within date block", hour_text))
            previous_hour = hour

            raw_value = row[station_column].strip()
            substituted = raw_value.endswith("*")
            value_text = raw_value.removesuffix("*")
            if raw_value in {"", "-"}:
                aqhi = float("nan")
            elif re.fullmatch(r"[1-9]|10[+]?", value_text):
                aqhi = 11 if value_text == "10+" else int(value_text)
            else:
                invalid_aqhi.append((location, raw_value))
                aqhi = float("nan")

            timestamp = datetime.combine(current_date, datetime.min.time()) + timedelta(hours=hour)
            rows.append((timestamp, aqhi, substituted, raw_value, location))


In [2]:
source_data = pd.DataFrame(
    rows, columns=["timestamp", "aqhi", "is_substituted", "raw_aqhi", "source"]
)
source_data["timestamp"] = pd.to_datetime(source_data["timestamp"]).dt.tz_localize("Asia/Hong_Kong")
source_data["aqhi"] = source_data["aqhi"].astype("float64")
source_data = source_data.sort_values("timestamp", kind="stable").reset_index(drop=True)

expected = pd.date_range(
    "2025-04-01 01:00", "2026-07-01 00:00", freq="h", tz="Asia/Hong_Kong"
)
observed = pd.DatetimeIndex(source_data["timestamp"])
duplicates = source_data.loc[observed.duplicated(keep=False)]
missing_hours = expected.difference(observed)
unexpected_hours = observed.difference(expected)
cleaned = source_data[["timestamp", "aqhi", "is_substituted"]]
quality = pd.Series({
    "Total hourly records": len(cleaned),
    "Duplicate timestamps": duplicates["timestamp"].nunique(),
    "Rows involved in duplicates": len(duplicates),
    "Missing hours": len(missing_hours),
    "Hours outside the expected grid": len(unexpected_hours),
    "Missing AQHI (NaN)": int(cleaned["aqhi"].isna().sum()),
    "Original AQHI blanks": int(source_data["raw_aqhi"].eq("").sum()),
    "Original AQHI dashes": int(source_data["raw_aqhi"].eq("-").sum()),
    "Substituted readings": int(cleaned["is_substituted"].sum()),
    "Date/hour format or sequence issues": len(date_hour_issues),
    "Unrecognized AQHI tokens": len(invalid_aqhi),
    "10+ readings encoded as 11": int(cleaned["aqhi"].eq(11).sum()),
})
substituted = quality["Substituted readings"]
all_pct = 100 * substituted / len(cleaned)
available_pct = 100 * substituted / cleaned["aqhi"].count()

print(quality.to_string())
print(f"Substituted: {all_pct:.2f}% of all rows; {available_pct:.2f}% of nonmissing readings")
for label, details in [
    ("Duplicate rows and original source lines", duplicates),
    ("Missing hours", missing_hours),
    ("Unexpected hours", unexpected_hours),
    ("Date/hour issues", date_hour_issues),
    ("Unrecognized AQHI", invalid_aqhi),
]:
    if len(details):
        print(label, details, sep="\n")

print(cleaned.head(10).to_string(index=False))
print(cleaned.dtypes.to_string())


Total hourly records                   10944
Duplicate timestamps                       0
Rows involved in duplicates                0
Missing hours                              0
Hours outside the expected grid            0
Missing AQHI (NaN)                        18
Original AQHI blanks                      12
Original AQHI dashes                       6
Substituted readings                    1083
Date/hour format or sequence issues        0
Unrecognized AQHI tokens                   0
10+ readings encoded as 11                15
Substituted: 9.90% of all rows; 9.91% of nonmissing readings
                timestamp  aqhi  is_substituted
2025-04-01 01:00:00+08:00   2.0           False
2025-04-01 02:00:00+08:00   2.0           False
2025-04-01 03:00:00+08:00   2.0           False
2025-04-01 04:00:00+08:00   2.0           False
2025-04-01 05:00:00+08:00   2.0           False
2025-04-01 06:00:00+08:00   2.0           False
2025-04-01 07:00:00+08:00   2.0           False
2025-04-01 08:0

## Export and handoff
AQHI is an ordinal scale. `10+` is encoded as 11 to preserve ordering and can be recovered by reversing that mapping. Blank cells and literal `-` are represented as NaN and exported as empty cells. The export retains all valid time rows, including any duplicates, for review. Missing timestamps are reported without adding rows.

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "tuen_mun_aqhi.csv"
# Report source issues first and prevent an export that silently drops invalid timestamps.
if date_hour_issues or invalid_aqhi:
    raise ValueError("Review the source issues printed above before exporting.")

cleaned.to_csv(csv_path, index=False, date_format="%Y-%m-%dT%H:%M:%S%z")
round_trip = pd.read_csv(csv_path)
round_trip["timestamp"] = pd.to_datetime(round_trip["timestamp"], utc=True).dt.tz_convert("Asia/Hong_Kong")
pd.testing.assert_frame_equal(round_trip, cleaned)

report = [
    "# Tuen Mun AQHI handoff",
    "",
    "Source: 15 Hong Kong EPD monthly English CSVs, April 2025 through June 2026.",
    "Reproduce with [01_data.ipynb](../../01_data.ipynb). Modeling input: [tuen_mun_aqhi.csv](tuen_mun_aqhi.csv).",
    "",
    "| Field | Type | Definition |",
    "|---|---|---|",
    "| timestamp | Timezone-aware datetime | Hong Kong hour endpoint (UTC+08:00). Source hour 24 is midnight the following day. |",
    "| aqhi | float64 | AQHI level. 11 encodes the official 10+ category. An empty CSV field parses as NaN. |",
    "| is_substituted | bool | True when EPD marked the reading with an asterisk, indicating a substitute station reading. |",
    "",
    "## Quality checks",
    "",
]
report.extend(f"- {label}: {count:,}" for label, count in quality.items())
report.extend([
    f"- Substituted share: {all_pct:.2f}% of all hourly rows and {available_pct:.2f}% of nonmissing AQHI.",
    f"- Timestamp coverage: {cleaned['timestamp'].iloc[0]} to {cleaned['timestamp'].iloc[-1]} inclusive.",
    "- The CSV contains hourly rows only. Original Daily Max rows remain in the source files.",
])
if len(missing_hours):
    report.extend(["", "Missing timestamps:", *[f"- {stamp.isoformat()}" for stamp in missing_hours]])
if len(duplicates):
    report.extend(["", "Duplicate rows (retained):", "```", duplicates.to_string(index=False), "```"])
(output_dir / "tuen_mun_aqhi_quality.md").write_text("\n".join(report) + "\n", encoding="utf-8")
print(f"Verified CSV round trip: {csv_path}")


Verified CSV round trip: data\processed\tuen_mun_aqhi.csv
